<a href="https://colab.research.google.com/github/Mokshitha08/RL/blob/main/RL_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def env_state():
  states=["S0","S1"]
  return states

In [ ]:
states=env_state()
states

['S0', 'S1']

In [ ]:
def env_actions():
  actions=["up","down","left","Right"]
  return actions

In [ ]:
actions=env_actions()
actions

['up', 'down', 'left', 'Right']

In [ ]:
def env_reward():
  reward={
    "move":-1,
    "goal":10,
    "obs":-1
  }
  return reward

In [ ]:
rewards=env_reward()
rewards

{'move': -1, 'goal': 10, 'obs': -1}

In [ ]:
def env_df():
  gama=0.9
  return gama

In [ ]:
gamma=env_df()
gamma

0.9

In [ ]:
def env_transition():
  transition={
      ("S0","up"):"S0",
      ("S0","down"):"S1",
      ("S0","left"):"S0",
      ("S0","Right"):"S0",
      ("S1","up"):"S0",
      ("S1","down"):"S1",
      ("S1","left"):"S1",
      ("S1","Right"):"G"
  }
  return transition

In [ ]:
transition = env_transition()
transition

{('S0', 'up'): 'S0',
 ('S0', 'down'): 'S1',
 ('S0', 'left'): 'S0',
 ('S0', 'Right'): 'S0',
 ('S1', 'up'): 'S0',
 ('S1', 'down'): 'S1',
 ('S1', 'left'): 'S1',
 ('S1', 'Right'): 'G'}

In [ ]:
def get_reward(state, action, next_state):
    # Reaching the terminal goal gives +10.
    if next_state == "G":
        return rewards["goal"]

    # All other transitions (including staying in place at a wall/boundary)
    # receive the normal movement reward of -1.
    return rewards["move"]

In [ ]:
# Example transition
state, action = "S1", "Right"
next_state = transition[(state, action)]
print(f"{state} --{action}--> {next_state}")
print("Reward:", get_reward(state, action, next_state))

S1 --Right--> G
Reward: 10


In [ ]:
V = {state: 0.0 for state in states}
V["G"] = 0.0

print("Initial state values:")
for state in states:
    print(f"V0({state}) = {V[state]}")
print(f"V(G) = {V['G']}")

Initial state values:
V0(S0) = 0.0
V0(S1) = 0.0
V(G) = 0.0


In [ ]:
theta = 1e-8
max_iterations = 100

iteration_history = [V.copy()]

for iteration in range(1, max_iterations + 1):
    old_V = V.copy()
    new_V = old_V.copy()

    # Synchronous update: every update uses old_V.
    for state in states:
        q_values = []

        for action in actions:
            next_state = transition[(state, action)]
            reward = get_reward(state, action, next_state)
            q_value = reward + gamma * old_V[next_state]
            q_values.append(q_value)

        new_V[state] = max(q_values)

    delta = max(abs(new_V[s] - old_V[s]) for s in states)
    V = new_V
    iteration_history.append(V.copy())

    print(
        f"Iteration {iteration:2d}: "
        + ", ".join(f"V({s}) = {V[s]:.6f}" for s in states)
        + f" | delta = {delta:.8f}"
    )

    if delta < theta:
        print(f"\nConverged after {iteration} iterations.")
        break
else:
    print("Warning: maximum iterations reached before convergence.")

Iteration  1: V(S0) = -1.000000, V(S1) = 10.000000 | delta = 10.00000000
Iteration  2: V(S0) = 8.000000, V(S1) = 10.000000 | delta = 9.00000000
Iteration  3: V(S0) = 8.000000, V(S1) = 10.000000 | delta = 0.00000000

Converged after 3 iterations.


In [ ]:
def calculate_q(state, action, state_values):
    next_state = transition[(state, action)]
    reward = get_reward(state, action, next_state)
    return reward + gamma * state_values[next_state]

Q = {}

for state in states:
    Q[state] = {}
    print(f"\nState {state}:")
    for action in actions:
        Q[state][action] = calculate_q(state, action, V)
        next_state = transition[(state, action)]
        print(
            f"Q*({state}, {action}) = "
            f"{get_reward(state, action, next_state)} + "
            f"{gamma} * V({next_state}) = {Q[state][action]:.6f}"
        )


State S0:
Q*(S0, up) = -1 + 0.9 * V(S0) = 6.200000
Q*(S0, down) = -1 + 0.9 * V(S1) = 8.000000
Q*(S0, left) = -1 + 0.9 * V(S0) = 6.200000
Q*(S0, Right) = -1 + 0.9 * V(S0) = 6.200000

State S1:
Q*(S1, up) = -1 + 0.9 * V(S0) = 6.200000
Q*(S1, down) = -1 + 0.9 * V(S1) = 8.000000
Q*(S1, left) = -1 + 0.9 * V(S1) = 8.000000
Q*(S1, Right) = 10 + 0.9 * V(G) = 10.000000


In [ ]:
optimal_policy = {}

for state in states:
    best_value = max(Q[state].values())
    # If actions tie, report all maximizing actions rather than hiding the tie.
    best_actions = [
        action for action, value in Q[state].items()
        if abs(value - best_value) < 1e-10
    ]
    optimal_policy[state] = best_actions

print("Optimal state values:")
for state in states:
    print(f"V*({state}) = {V[state]:.6f}")

print("\nOptimal policy:")
for state in states:
    print(f"pi*({state}) = {optimal_policy[state]}")

Optimal state values:
V*(S0) = 8.000000
V*(S1) = 10.000000

Optimal policy:
pi*(S0) = ['down']
pi*(S1) = ['Right']
